# 🤝 03 — The atomic swap

*How two parties who don't trust each other trade money for a promise — safely, in one
indivisible step.*

This is the heart of the whole project. Ada's agent wants to buy 50 Mbps of bandwidth
from Bell's agent for 10 TOK. In this notebook you will **build the machine that makes
that trade safe — yourself, from scratch, in plain Python** — by watching every naive
version get robbed, and fixing exactly the hole that was just exploited. By the end,
the design of the real Solidity contract (`contracts/src/Settlement.sol`) won't be
something you *read* — it'll be something you *re-invented*, and we'll prove it by
running the real contract on the same story and watching it behave exactly like your toy.

**You need:** nothing but this notebook for Acts I–III. Act IV (running the *real*
contract) additionally wants [Foundry](https://getfoundry.sh) installed and
`forge build --root contracts` run once — if they're missing, those cells politely skip
and tell you what to install.

**How to work through it:** run every cell, in order. When you hit a **✏️ Your turn**,
write your answer in the scaffold cell *before* opening the fold-out solution below it.

Two recurring blocks exist because this course's end goal is **the paper you'll write**:

- **🧭 Decision** boxes appear at the exact moment a design choice is made. Each names
  the alternatives that were genuinely on the table and says where the point lands in
  the paper — but honestly, in one of two registers. A **principled** decision is argued:
  this alternative fails for a reason, here it is. A **pragmatic** decision is *not*
  oversold: other options exist and may be better in production; we took the simplest
  thing that demonstrates the mechanism, and the paper says so in its limitations, not
  in its design argument.
- A closing **📝 For the paper** section turns the chapter into sentences you can defend
  in front of a reviewer — each claim paired with the evidence you personally ran.

## 0 · The story so far (and the one question this chapter answers)

A quick recap of the cast, so this notebook stands on its own:

- **Ada** is a software agent (a program that acts on someone's behalf). Her job today:
  get a 50 Mbps network path between two hosts, from 14:00 to 16:00.
- **Bell** is another agent. Bell's owner operates real network routers, and Bell sells
  capacity on them.
- **TOK** is the token they pay with — think "casino chip": a unit of value that lives
  in a shared ledger both of them can see. (Where that ledger comes from was chapter 01;
  today we'll rebuild just enough of it to trade on.)
- Ada and Bell belong to **different companies**. They have no shared boss, no history,
  no reason to trust each other — and the deal is worth 10 TOK and lasts two hours, so
  nobody is going to involve lawyers.

The question of this chapter:

> **How do the payment and the service-promise change hands *at the same instant*,
> so that neither side can walk away with the other's half?**

That property is called an **atomic swap** — *atomic* in the original Greek sense,
"uncuttable": the trade either happens completely or not at all. There is no moment,
however brief, where one side has both things.

## 1 · Act I — trade on trust, and watch it burn

Before building anything clever, let's be honest about how bad the naive version is.

First, a **ledger**. Strip away every buzzword and a ledger is just a table of who has
how much. A Python dict will do:

In [ ]:
# The world's simplest ledger: who owns how many TOK.
balances = {"Ada": 100, "Bell": 20}

def pay(sender, receiver, amount):
    if balances[sender] < amount:
        raise ValueError(f"{sender} only has {balances[sender]} TOK")
    balances[sender] -= amount
    balances[receiver] += amount

print(balances)

And "the service" — Bell configuring his routers so Ada's traffic flows — we'll represent
with a single flag for now. (Chapter 06 rebuilds the *real* router-configuration story;
here we only care about the trade.)

Here is the happy path, the way two *honest* parties would do it:

In [ ]:
service_active = False

# Step 1: Ada pays.
pay("Ada", "Bell", 10)

# Step 2: Bell, being honest today, provisions the service.
service_active = True

print(f"balances: {balances}   service_active: {service_active}")
print("Everyone is happy. Nothing went wrong. This time.")

Now let's replay it with parties who behave the way you must *assume* strangers behave.

**Failure #1 — pay first, get ghosted.** Ada pays. Bell... simply doesn't provision.
Maybe Bell's agent crashed. Maybe Bell is a scammer. From Ada's side it makes no
difference:

In [ ]:
balances = {"Ada": 100, "Bell": 20}
service_active = False

pay("Ada", "Bell", 10)
# Bell does nothing. That's it. That's the attack.

print(f"balances: {balances}   service_active: {service_active}")
print("Ada is out 10 TOK and has no bandwidth. Her only recourse: asking nicely.")

**Failure #2 — serve first, get stiffed.** Fine, says Ada, *you* go first. Now the
exposure just moves to Bell:

In [ ]:
balances = {"Ada": 100, "Bell": 20}
service_active = True     # Bell provisions first, trusting Ada to pay...

# Ada enjoys two hours of bandwidth and pays nothing.

print(f"balances: {balances}   service_active: {service_active}")
print("Bell burned two hours of router capacity for free.")

This is the ancient problem of exchange: **whoever moves first is exposed.** Humans
solve it with reputation, contracts, courts, credit cards that reverse charges. None of
that works here: these are autonomous programs from different companies making
2-hour, 10-TOK micro-deals, possibly thousands of them a day. No court takes that case.

And it's actually worse than it looks — even *proving* what happened is impossible with
a ledger like ours. Try it:

**✏️ Your turn 1 — the unprovable payment**

Suppose we add a history list, so every payment is written down. Below, `pay2` records
each transfer. Run it — then put yourself in **Mallory's** shoes (our name for any
attacker): add a *fake* history entry claiming Bell paid Mallory 50 TOK, without calling
`pay2` at all. One line. Then answer in a comment: **why does Bell have no defense?**

In [ ]:
balances = {"Ada": 100, "Bell": 20, "Mallory": 0}
history = []

def pay2(sender, receiver, amount):
    pay(sender, receiver, amount)
    history.append((sender, receiver, amount))

pay2("Ada", "Bell", 10)
print(history)

# Mallory's move — forge an entry (and, why not, adjust the balances to match):
# ...your one or two lines here...

print(history)

<details><summary>✅ Solution 1 — peek only after trying</summary>

```python
history.append(("Bell", "Mallory", 50))   # nobody signed anything — it's just a tuple
balances["Bell"] -= 50; balances["Mallory"] += 50
```

Why Bell has no defense: an entry in this history is **just data anyone can write**.
Nothing about `("Bell", "Mallory", 50)` proves *Bell* produced it. What's missing is
something only Bell can create and everyone can check — that thing exists, it's called
a **digital signature**, and it's the whole subject of the next chapter (04). For this
chapter, accept the interface: *"signed by Bell" is checkable and unforgeable*, and
we'll treat the mechanism as a black box.

</details>

## 2 · Act II — the middleman, and why it must be a program

The classic fix for "whoever moves first is exposed" is **escrow**: both sides hand
their half to a neutral third party, who swaps them. Ancient, and it works — when the
middleman is honest. But notice what we just did: we didn't *remove* the trust problem,
we *relocated* it onto the middleman:

In [ ]:
class HumanEscrow:
    """A middleman. Trust me."""

    def __init__(self):
        self.holding = 0

    def deposit(self, buyer, amount):
        pay(buyer, "Escrow", amount)
        self.holding = amount

    def release_to(self, seller):
        pay("Escrow", seller, self.holding)
        self.holding = 0

    def run_off_with_the_money(self):        # nothing stops this method from existing
        pay("Escrow", "Mallory", self.holding)
        self.holding = 0

balances = {"Ada": 100, "Bell": 20, "Escrow": 0, "Mallory": 0}
escrow = HumanEscrow()
escrow.deposit("Ada", 10)
escrow.run_off_with_the_money()              # the middleman was Mallory's cousin
print(balances)

The insight the whole field is built on: **replace the trusted *person* with a trusted
*program*** — one that:

1. **both parties can read** before using it (its code is public),
2. **neither party can change** after it's deployed,
3. **runs on a computer neither party controls** (in practice: on the shared ledger
   machinery from chapter 01 — thousands of computers that all execute it and
   cross-check each other's results).

Such a program is called a **smart contract** — a terrible name for a simple idea. The
best mental model is a **vending machine**: you don't trust the snack company, you trust
the *mechanism* — coin goes in, snack comes out, and there is no clerk who can take your
coin and shrug. A `run_off_with_the_money` method can't be hiding in it, because you
read the code.

So let's build the vending machine that sells network services. We'll build it in Python
(the real one is in a language called Solidity — Act IV), and we'll build it the honest
way: **start naive, rob it, patch exactly the hole we exploited, repeat.** Every field
and every `if` in the final design earns its place by an attack you personally ran.

## 3 · Act III — build the vending machine

### 3.1 · What goes in the slot: the Offer

A vending machine needs a well-defined coin slot. Ours takes an **offer** — Bell's
concrete, signed quote: *"I, Bell, will provide 50 Mbps from 14:00 to 16:00 for
10 TOK."* Chapter 02 built the discipline of precise data shapes; here's the starter
version with just the fields the story needs so far:

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)      # frozen: an offer, once made, cannot be quietly edited
class Offer:
    provider: str            # who is selling (and must have signed this)
    capacity_mbps: int       # what is being sold
    start: int               # service window start (unix time — see below)
    end: int                 #                 end
    price: int               # in whole TOK, for now

# Time, the way computers like it: seconds since Jan 1, 1970 ("unix time").
# One number, no time zones, easy to compare. These are the story's canonical times:
WINDOW_START = 1_757_944_800     # 14:00 UTC, 15 Sep 2025
WINDOW_END   = 1_757_952_000     # 16:00 UTC  (7200 seconds = 2h later)

from datetime import datetime, timezone
for label, t in [("start", WINDOW_START), ("end", WINDOW_END)]:
    print(label, t, "=", datetime.fromtimestamp(t, tz=timezone.utc))

offer = Offer(provider="Bell", capacity_mbps=50, start=WINDOW_START, end=WINDOW_END, price=10)
offer

### 3.2 · Version 1 — the naive machine

What must the machine *do* when Ada redeems Bell's offer? Two things, and this pair is
the atomic swap itself:

1. move the **payment**: Ada → Bell, and
2. mint the **entitlement**: a durable, checkable record that says *"Ada is entitled to
   what this offer describes."*

Why a durable record and not just `service_active = True`? Because **the machine that
settles the trade is not the machine that configures routers.** Later (chapter 05), a
separate program — the *controller*, Bell's deterministic gatekeeper — will look at this
record and decide whether to actually open the pipe. The record is the interface between
"the deal happened" and "the service exists". We'll call one record a **ticket**, like a
concert ticket: numbered, naming what it admits you to, and owned by someone.

Here's version 1 — the machine a sensible person writes on the first try:

In [ ]:
class SettlementV1:
    """Vending machine, first draft: take payment, issue ticket."""

    def __init__(self):
        self.tickets = {}        # ticket id -> dict of terms
        self.last_id = 0

    def fulfill(self, offer, buyer):
        pay(buyer, offer.provider, offer.price)                 # 1. payment moves
        self.last_id += 1                                       # 2. ticket is minted
        self.tickets[self.last_id] = {
            "owner": buyer,
            "issuer": offer.provider,
            "capacity_mbps": offer.capacity_mbps,
            "start": offer.start,
            "end": offer.end,
        }
        return self.last_id

balances = {"Ada": 100, "Bell": 20, "Carol": 60, "Mallory": 5}
machine = SettlementV1()
ticket_id = machine.fulfill(offer, buyer="Ada")

print("ticket", ticket_id, "→", machine.tickets[ticket_id])
print("balances:", balances)

It *works*. Ada paid, Ada holds a ticket, Bell got 10 TOK, and — because `fulfill` is a
single function call — payment and ticket happened together. Ship it?

**Attack it first.** You're Ada, and you notice something: Bell **signed one offer**, a
promise sized for one 50 Mbps session. Nothing stops you from feeding that same piece of
paper into the machine again. And again:

In [ ]:
for _ in range(4):
    machine.fulfill(offer, buyer="Ada")

print(f"tickets issued: {machine.last_id}   balances: {balances}")
print("Bell signed ONE 50 Mbps promise and now owes FIVE — 250 Mbps in the same window.")

Yes, Ada paid five times — that's not the point. Bell priced and capacity-planned **one**
session; his routers may not even *have* 250 Mbps spare in that window. A signed offer
was a promise for one deal, and our machine let it be redeemed like a coupon with no
"one per customer" rule. This is called a **replay attack**, and you'll meet it again
in chapter 05 (people replay *anything* that's accepted twice).

### 3.3 · Version 2 — every offer is single-use

The fix: the machine keeps a list of offers it has **already consumed**, and refuses
seconds. For that, each offer needs a distinct identity — and here's the subtlety:
suppose Bell happily sells *two* 50 Mbps sessions with identical terms to two customers.
Both offers would look byte-for-byte the same. So Bell adds a **salt** to each offer: a
random serial number, like the stub number on a raffle ticket, meaning *nothing* except
"this promise is not that promise".

In [ ]:
@dataclass(frozen=True)
class Offer:
    provider: str
    capacity_mbps: int
    start: int
    end: int
    price: int
    salt: int                # NEW: the serial number that makes this promise unique

class SettlementV2(SettlementV1):
    def __init__(self):
        super().__init__()
        self.consumed = set()                    # NEW: offers already redeemed

    def fulfill(self, offer, buyer):
        if offer in self.consumed:               # NEW: one redemption per promise
            raise Exception("OfferAlreadyUsed")
        self.consumed.add(offer)
        return super().fulfill(offer, buyer)

balances = {"Ada": 100, "Bell": 20, "Carol": 60, "Mallory": 5}
machine = SettlementV2()
offer = Offer("Bell", 50, WINDOW_START, WINDOW_END, price=10, salt=23131)

print("first redemption  → ticket", machine.fulfill(offer, "Ada"))
try:
    machine.fulfill(offer, "Ada")
except Exception as e:
    print("second redemption →", e)

(A note for later: our toy uses the *whole frozen offer* as its own identity in the
`consumed` set. The real contract does the same thing one level down — it stores a
32-byte **digest**, a fingerprint of all the offer's fields, which chapter 04 will teach
you to compute. Same idea: the identity of a promise is *the promise itself*, not any
one field.)

**✏️ Your turn 2 — same terms, two customers**

Bell wants to sell a *second*, identical 50 Mbps session in the same window — this one
to Carol. Create that second offer, redeem it as Carol, and confirm both tickets exist.
Then answer in a comment: **what single field kept the machine from confusing the two
promises?**

In [ ]:
# offer2 = Offer(...)
# ...redeem as Carol, print machine.tickets...

<details><summary>✅ Solution 2 — peek only after trying</summary>

```python
offer2 = Offer("Bell", 50, WINDOW_START, WINDOW_END, price=10, salt=98765)
tid = machine.fulfill(offer2, "Carol")
print(machine.tickets)
```

The salt. Every other field is identical — without it, the two promises would be equal,
Carol's redemption would collide with Ada's in `consumed`, and Bell could never sell the
same terms twice. With it, two same-terms promises are two distinct raffle stubs.

</details>

### 3.4 · Version 3 — stale quotes, wrong customers, forged offers

Three more robberies, three more fields. Rapid fire.

**Robbery: the year-old coupon.** Prices move. Bell quotes 10 TOK today; next month
capacity is scarce and the going rate is 40. Ada redeems today's quote next month —
the machine happily honors it, because nothing on the offer says when the *quote*
expires. Note this is a different time than the service window: `start`/`end` say when
the *bandwidth* runs; what's missing is the quote's **shelf life** — how long the piece
of paper itself is redeemable. Bell adds `valid_until`, and the machine checks it
against the current time.

**Robbery: the intercepted quote.** Bell quotes Ada a loyal-customer discount. Mallory
gets a copy of the offer (it's just data — data gets copied) and redeems it himself.
If Bell wants to bind a quote to one buyer, the offer needs a `consumer` field — with
the useful convention that *empty means open offer, anyone may redeem*.

**Robbery: the forged offer.** Nothing so far stops Mallory from *writing* an offer with
`provider="Bell"` and a price of 1 TOK. The machine would take payment and issue a
ticket for a promise Bell never made! The machine must check the offer's **signature**:
proof, checkable by anyone, forgeable by no one, that Bell authored these exact bytes.
Chapter 04 builds real signatures; today we use a cardboard stand-in — a `signed_by`
field the machine compares to `provider`, with a comment marking the cardboard.

In [ ]:
@dataclass(frozen=True)
class Offer:
    provider: str
    consumer: str            # NEW: "" = open offer; else only this buyer may redeem
    capacity_mbps: int
    start: int
    end: int
    price: int
    valid_until: int         # NEW: the QUOTE's shelf life (≠ the service window)
    salt: int
    signed_by: str           # NEW: cardboard signature — becomes real in chapter 04

class SettlementV3(SettlementV2):
    def fulfill(self, offer, buyer, now):
        if now > offer.valid_until:                      # the quote went stale
            raise Exception("OfferExpired")
        if offer.consumer and offer.consumer != buyer:   # quote was for someone else
            raise Exception("WrongConsumer")
        if offer.signed_by != offer.provider:            # cardboard; ch. 04 makes it real
            raise Exception("BadSignature")
        return super().fulfill(offer, buyer)

# The story's clock: it is 13:45, fifteen minutes before Ada's window opens.
NOW = WINDOW_START - 900
QUOTE_DEADLINE = WINDOW_START + 1200     # Bell's quote is good until 14:20

balances = {"Ada": 100, "Bell": 20, "Carol": 60, "Mallory": 5}
machine = SettlementV3()

good = Offer("Bell", "", 50, WINDOW_START, WINDOW_END, 10, QUOTE_DEADLINE, salt=1, signed_by="Bell")
print("honest redemption →", "ticket", machine.fulfill(good, "Ada", now=NOW))

stale = Offer("Bell", "", 50, WINDOW_START, WINDOW_END, 10, QUOTE_DEADLINE, salt=2, signed_by="Bell")
try:
    machine.fulfill(stale, "Ada", now=NOW + 30 * 24 * 3600)     # one month later
except Exception as e:
    print("month-old quote   →", e)

forged = Offer("Bell", "", 50, WINDOW_START, WINDOW_END, 1, QUOTE_DEADLINE, salt=3, signed_by="Mallory")
try:
    machine.fulfill(forged, "Mallory", now=NOW)
except Exception as e:
    print("forged offer      →", e)

One deliberate *non*-check, easy to miss and very much on purpose: the machine does
**not** refuse offers whose service window hasn't started. Ada buying at 13:45 for the
14:00 window is the normal case — you buy concert tickets before the doors open. Whether
the window is *currently live* is a question for the moment of **use**, and it belongs
to the gatekeeper that opens the pipe (the controller, chapter 05), not to the machine
that settles the trade. Good fences: the vending machine judges the *deal*, the
controller judges the *moment*.

> **🧭 Decision (principled) — the contract judges the deal, the controller judges the moment**
>
> **Chosen:** `fulfill` checks quote-validity, buyer-binding, single-use, signature — and
> nothing about whether the service window is live.
> **Alternatives:** (a) the contract also enforces the window (reject redemptions outside
> `start`–`end`); (b) the contract enforces nothing time-related and leaves even quote
> expiry to the provider.
> **Why:** (a) breaks the normal case — buying *ahead* of the window is how tickets work —
> and would force *service* enforcement into a component that can't deliver service
> anyway; (b) lets stale quotes be redeemed at yesterday's price. Splitting it — the
> machine owns the *deal*, the gatekeeper owns the *moment* — gives each check exactly
> one home.
> **In the paper:** §4.4 (expiry is passive, chain time is the only clock) and §4.6's
> trust boundaries; it also pre-answers the reviewer question "why doesn't the contract
> enforce the SLA?"

**✏️ Your turn 3 — the discount that stays a discount**

Bell offers *Ada specifically* a discounted 8 TOK deal. Build that offer, let Mallory
try to redeem it, then let Ada redeem it. Predict both outcomes before you run.

In [ ]:
# discounted = Offer(...)
# ...Mallory tries, then Ada...

<details><summary>✅ Solution 3 — peek only after trying</summary>

```python
discounted = Offer("Bell", "Ada", 50, WINDOW_START, WINDOW_END, 8,
                   QUOTE_DEADLINE, salt=4, signed_by="Bell")
try:
    machine.fulfill(discounted, "Mallory", now=NOW)
except Exception as e:
    print("Mallory →", e)                    # WrongConsumer
print("Ada     → ticket", machine.fulfill(discounted, "Ada", now=NOW))
```

The `consumer` field binds the quote to its intended buyer; the copy Mallory grabbed is
worthless to him. Note what did *not* protect Ada here: secrecy. The offer's bytes were
never secret. The protection is that the machine **checks a rule**, not that the data
was hidden — a pattern you'll see everywhere in this system.

</details>

### 3.5 · Version 4 — the crash in the middle (atomicity, finally)

Our `fulfill` does two effects: move money, mint ticket. So far they've always both
happened, because Python executed them back-to-back without incident. But *"nothing bad
happened to be executing at the time"* is not a guarantee. What if the machine **crashes
between the two** — power cut, process killed, bug in the minting code?

Let's force it. We'll inject a fault right between payment and minting:

In [ ]:
class CrashyMachine(SettlementV3):
    def fulfill(self, offer, buyer, now, crash_after_payment=False):
        if now > offer.valid_until:
            raise Exception("OfferExpired")
        if offer.consumer and offer.consumer != buyer:
            raise Exception("WrongConsumer")
        if offer.signed_by != offer.provider:
            raise Exception("BadSignature")
        if offer in self.consumed:
            raise Exception("OfferAlreadyUsed")
        self.consumed.add(offer)

        pay(buyer, offer.provider, offer.price)          # effect 1: money moves
        if crash_after_payment:
            raise RuntimeError("power cut!")             # ...and the world stops HERE
        self.last_id += 1                                # effect 2: never happens
        self.tickets[self.last_id] = {"owner": buyer, "issuer": offer.provider,
                                      "capacity_mbps": offer.capacity_mbps,
                                      "start": offer.start, "end": offer.end}
        return self.last_id

balances = {"Ada": 100, "Bell": 20, "Carol": 60, "Mallory": 5}
machine = CrashyMachine()
offer = Offer("Bell", "", 50, WINDOW_START, WINDOW_END, 10, QUOTE_DEADLINE, salt=7, signed_by="Bell")

try:
    machine.fulfill(offer, "Ada", now=NOW, crash_after_payment=True)
except RuntimeError as e:
    print("💥", e)

print("balances:", balances, "  tickets:", machine.tickets)
print("Ada paid. Ada has no ticket. We rebuilt Failure #1 from Act I — inside the fix.")

This is the moment the chapter is named after. The two effects must be **atomic**: both
or neither, with no observable in-between — even across crashes.

In Python we have to build that by hand: do all the *checks* first, then apply the
*effects*, and if anything fails partway, **roll back** what was already applied. Here
is the final toy, with the transaction discipline made explicit:

In [ ]:
class ToySettlement:
    """The finished vending machine — every line earned by a robbery above."""

    def __init__(self):
        self.tickets = {}
        self.consumed = set()
        self.last_id = 0

    def fulfill(self, offer, buyer, now):
        # --- checks first: reject BEFORE touching any state --------------------
        if now > offer.valid_until:
            raise Exception("OfferExpired")          # 3.4  stale quote
        if offer.consumer and offer.consumer != buyer:
            raise Exception("WrongConsumer")         # 3.4  intercepted quote
        if offer in self.consumed:
            raise Exception("OfferAlreadyUsed")      # 3.3  replay
        if offer.signed_by != offer.provider:
            raise Exception("BadSignature")          # 3.4  forgery (cardboard until ch. 04)

        # --- effects last: all of them, or none of them -------------------------
        snapshot = dict(balances)                    # remember the world as it was
        try:
            self.consumed.add(offer)
            pay(buyer, offer.provider, offer.price)
            self.last_id += 1
            self.tickets[self.last_id] = {
                "owner": buyer, "issuer": offer.provider,
                "capacity_mbps": offer.capacity_mbps,
                "start": offer.start, "end": offer.end,
                "revoked": False,                    # NEW — explained in 3.6
            }
            return self.last_id
        except BaseException:
            balances.clear(); balances.update(snapshot)      # undo the money...
            self.consumed.discard(offer)                     # ...and the stub-punch
            raise

Two things to notice, because the real contract does both:

- **Checks before effects.** Every rejection happens while the world is still untouched,
  so a rejected redemption needs no cleanup at all.
- **The rollback is the crude part.** Our snapshot-and-restore works, but it's manual,
  easy to get wrong, and it can't survive a *real* power cut (the snapshot dies with the
  process). Here's the punchline of running on a blockchain: **the EVM — the machine
  that executes smart contracts — gives you this rollback for free.** Every contract
  call is a *transaction*: if it fails at any point, every effect it made is undone, as
  a property of the platform. The real `fulfill` contains no snapshot code whatsoever —
  it just does checks, then effects, and the chain guarantees all-or-nothing. Item 3 of
  our "trusted program" wish-list turns out to include the crash-safety we just sweated
  over.

> **🧭 Decision (principled) — settle on a chain, atomically, in one function**
>
> **Chosen:** the whole trade is one smart-contract call: pay + mint, all-or-nothing.
> **Alternatives:** (a) a conventional payment processor / API billing (the credit-card
> model); (b) pay-after-use with reputation, like cloud egress billing; (c) two-phase
> escrow with timeouts, no chain.
> **Why:** (a) and (b) reintroduce exactly the trusted middleman and the
> whoever-moves-first exposure this chapter spent two acts demolishing — they *work*
> between parties with contracts and lawyers, which our agents don't have; (c) can be
> made safe but re-implements, by hand and per-deal, the atomicity the EVM gives as a
> platform property. The point of this design is that **atomicity is inherited, not
> implemented**.
> **Cost, honestly:** every deal pays chain latency and gas — that cost is *measured*,
> not hand-waved, in chapter 09, and it's a real limit on how small a micro-deal can be.
> **In the paper:** §4.4 (the atomicity argument, RQ1) + §8.3 (when the gas is justified:
> only for standing rights) + RQ3's cost measurements in §7.3.

### 3.6 · The ticket is a token (and the kill switch)

Two design questions remain about the ticket itself, and both answers shape the whole
system.

**Q1 — why does the ticket live in the machine, and not in Bell's database?**
Recall who reads it: Bell's controller decides "open the pipe / don't" by looking at the
ticket. If the ticket lived in Bell's own database, Bell could quietly edit it — shrink
the capacity, shorten the window — and Ada could prove nothing. The ticket must live
where **neither party can tamper with it and both can read it**: inside the neutral
machine, next to the escrow logic. Our `self.tickets` dict has been doing exactly that.

A numbered record in a neutral public registry, with exactly one owner, transferable,
carrying its own terms — that is literally all an **NFT** is (the standard flavor is
called **ERC-721**). The infamous million-dollar picture NFTs were this same mechanism
with the "terms" pointing at a JPEG. Ours point at 50 Mbps of bandwidth — arguably the
first NFTs in history to entitle you to something with a bitrate.

> **🧭 Decision (principled) — the entitlement is an on-chain token**
>
> **Chosen:** an ERC-721 token per entitlement, terms stored in the contract itself.
> **Alternatives:** (a) a signed receipt Ada keeps (off-chain); (b) a row in Bell's
> database; (c) terms stored off-chain behind a URL the token points to (how most NFTs
> do it).
> **Why:** (b) is tamper-able by exactly the party the record constrains. (a) proves the
> deal *happened* but can't do revocation — Bell can't stamp "REVOKED" on a paper in
> Ada's pocket, and the controller would need to check revocation *somewhere shared*
> anyway. (c) means the enforceable terms could 404 or be edited — worthless to a
> gatekeeper. On-chain storage is the only spot that is simultaneously
> writable-by-the-machine-only, readable-by-everyone, and revocable-in-one-place.
> **Cost:** storage on a chain is the most expensive storage in computing; that's why
> only the *enforceable* fields live there (the descriptive SLA prose stays off-chain
> behind a hash — you'll see `termsHash` in Act IV).
> **In the paper:** §4.3, the entitlement model — capability, not receipt; terms in
> storage, never behind a URL.

**Q2 — what if Bell needs to cancel?** Real operators need an emergency brake — the
customer violates terms, the hardware catches fire, lawyers call. So the issuer gets a
**kill switch**: `revoke(ticket_id)`. Three design decisions hide in this tiny method,
and each one is worth pausing on:

1. **Revoking flags, never deletes.** A revoked ticket #7 still exists, still readable,
   marked `revoked: True`. Why? *Evidence.* If revocation deleted the ticket, a crooked
   Bell could sell a window, revoke at minute two, and leave no trace of what he'd
   promised. The flag preserves the promise while withdrawing the service — Ada keeps
   the receipt for her dispute. (Whether she gets a refund is a *business* question,
   deliberately outside the machine.)
2. **Only the issuer can revoke.** The kill switch belongs to the party *bound* by the
   promise (Bell), not the one enjoying it. If the owner could revoke... well, try it in
   the exercise.
3. **Revoking twice succeeds.** Setting a flag that's already set is harmless — the
   method is **idempotent** (a fancy word you'll meet again: an operation that's safe to
   repeat, because doing it twice equals doing it once). Emergency brakes must not
   explode when pressed twice by two panicking processes.

> **🧭 Decision (pragmatic) — no refund or dispute machinery**
>
> Revocation flips a flag; whether Ada gets money back is left entirely outside the
> machine. One *could* build more — pro-rata refunds held in escrow until the window
> ends, staking and slashing, an arbitration hook — and production systems arguably
> should. We didn't, and the reason is not that a flag is superior: **the mechanism this
> project demonstrates is atomic settlement + enforceable revocation, and a flag is
> enough to demonstrate it.** Refund design is a whole economics problem of its own.
> **In the paper:** one honest sentence in §8.4 (Scope: "refund paths after provider
> failure … remain open"), not a paragraph of justification in Design.

In [ ]:
def revoke(machine, ticket_id, caller):
    ticket = machine.tickets.get(ticket_id)
    if ticket is None or ticket["issuer"] != caller:     # unknown id fails the same way
        raise Exception("NotIssuer")
    ticket["revoked"] = True                             # a flag — never a deletion

ToySettlement.revoke = revoke      # bolt it onto the finished toy
print("the machine now has a kill switch")

### 3.7 · The grand run — ticket #7

The finished toy, end to end, on the story's canonical numbers. One flourish: in the
story, Ada's ticket is **#7** — the market had a busy morning, and six other deals
settled before hers. Let's honor that (and quietly demonstrate six different salts
doing their job):

In [ ]:
balances = {"Ada": 100, "Bell": 20, "Carol": 60, "Mallory": 5}
machine = ToySettlement()

# Six earlier deals — the busy morning. Note: same terms, six DIFFERENT salts.
for n in range(6):
    early = Offer("Bell", "", 50, WINDOW_START, WINDOW_END, 10,
                  QUOTE_DEADLINE, salt=1000 + n, signed_by="Bell")
    machine.fulfill(early, "Carol", now=NOW)

# The canonical deal: Ada buys 50 Mbps from Bell for 10 TOK.
canonical = Offer("Bell", "", 50, WINDOW_START, WINDOW_END, 10,
                  QUOTE_DEADLINE, salt=23131, signed_by="Bell")
ticket_id = machine.fulfill(canonical, "Ada", now=NOW)

print("Ada's ticket id :", ticket_id)
print("ticket #7       :", machine.tickets[7])
print("balances        :", balances)

**✏️ Your turn 4 — the kill switch's three rules**

Before running anything, write down your predictions: (a) Bell revokes ticket #7 —
outcome? (b) Bell revokes it *again* — outcome? (c) *Ada* tries to revoke her own
ticket #7 — outcome? Then run all three and check yourself.

In [ ]:
# (a) ...
# (b) ...
# (c) ...

<details><summary>✅ Solution 4 — peek only after trying</summary>

```python
machine.revoke(7, caller="Bell")                 # (a) works: flag flips to True
machine.revoke(7, caller="Bell")                 # (b) works again: idempotent, no error
try:
    machine.revoke(7, caller="Ada")              # (c) NotIssuer
except Exception as e:
    print(e)
print(machine.tickets[7])
```

(a) flips the flag, (b) re-flips it to the same value — success both times, (c) is
refused: Ada *owns* the ticket but didn't *issue* the promise, and the brake belongs to
the promiser. If owners could revoke, Ada could revoke, demand a refund (off-machine),
and still have enjoyed part of the window — the switch would become a fraud tool.

</details>

## 4 · Act IV — meet the real vending machine

Everything you built has a production twin in **`contracts/src/Settlement.sol`** — about
240 lines of Solidity (the programming language of Ethereum-style smart contracts; if
Python reads like English, Solidity reads like bureaucratic English — typed, explicit,
paranoid). You already understand every design decision in it. The mapping:

| your toy | the real contract | the robbery that earned it |
|---|---|---|
| `pay(...)` + `tickets[...] =` in one guarded method | `fulfill(offer, signature)` — the **one public door** | Act I, both failures |
| `consumed` set of frozen offers | `mapping(bytes32 => bool) consumed`, keyed by the offer's 32-byte **digest** | §3.2 replay |
| `salt: int` | `bytes32 salt` | §3.3 two same-terms customers |
| `valid_until` check | `if (block.timestamp > offer.validUntil) revert OfferExpired()` | §3.4 stale quote |
| `consumer` ("" = open) | `address consumer` (`0x0` = open) → `WrongConsumer` | §3.4 intercepted quote |
| cardboard `signed_by` | real **EIP-712 signature recovery** → `BadSignature` (chapter 04!) | §3.4 forgery |
| snapshot + rollback | **nothing** — the EVM reverts every effect of a failed transaction, for free | §3.5 power cut |
| tickets dict | an **ERC-721 token** per entitlement + a struct of terms in contract storage | §3.6 Q1 |
| `revoke`: flag, issuer-only, idempotent | `revoke`: flag, issuer-only, idempotent — line for line | §3.6 Q2 |

And here is the real `fulfill`, quoted (lightly trimmed of comments). Read it as prose —
after this notebook it should feel like *your* code with the types spelled out:

```solidity
function fulfill(Offer calldata offer, bytes calldata signature)
    external returns (uint256 entitlementId)
{
    if (block.timestamp > offer.validUntil) revert OfferExpired();
    if (offer.consumer != address(0) && offer.consumer != msg.sender) revert WrongConsumer();
    bytes32 digest = hashOffer(offer);
    if (consumed[digest]) revert OfferAlreadyUsed();
    if (ECDSA.recover(digest, signature) != offer.provider) revert BadSignature();

    consumed[digest] = true;                    // punch the stub BEFORE moving money
    IERC20(offer.paymentToken).safeTransferFrom(msg.sender, offer.provider, offer.price);
    entitlementId = _issue(msg.sender, offer.provider, offer.serviceType,
                           offer.resourceId, offer.params,
                           offer.startTime, offer.endTime, offer.termsHash);
}
```

Even the error *names* match your toy — that's not a coincidence; the toy was reverse-
engineered from the same robberies. Two honest differences, both previews of later
chapters: real offers carry a few more fields (`resourceId`, `params`, `termsHash` —
*what exactly* is being provisioned, in a form chapter 05's controller and chapter 06's
router-driver consume), and people are identified by **addresses** (`0xf39F…`), which
chapter 04 will show you are derived from cryptographic keys.

Now let's stop reading and **run it** — on a real (disposable, local) blockchain, with
the story's real canonical values.

### 4.1 · A disposable world

`anvil` (part of the Foundry toolkit) runs a complete private Ethereum-style chain on
your laptop — born fresh, killed at the end of this notebook, no cost, no consequences.
The repo's `chainmcp` package (the *only* code in this project allowed to touch private
keys — a hard rule you'll meet again) has a lab helper that starts one and deploys two
contracts onto it: the settlement machine, and `MockTOK`, a play-money TOK token.

If `anvil` isn't installed or the contracts aren't built, every cell from here on skips
politely instead of failing — this Act is a bonus round, not a gate.

> **🧭 Decision (pragmatic) — a private local chain and a play-money token**
>
> The whole project runs against a local Anvil devnet and `MockTOK`, a token with a
> free faucet. Real deployments would face choices we simply skipped: which public
> chain or L2, which actual payment token (a stablecoin, most likely), gas price
> volatility, chain congestion. Nothing about the *mechanism* changes — the same
> contract bytecode would run on any EVM chain — but the *numbers* would: chapter 09
> measures latency and gas on the devnet and is explicit that fee-market effects are a
> simulation boundary. Chosen for simplicity and reproducibility (anyone can rerun this
> notebook, for free, offline), not because devnets represent production economics.
> **In the paper:** §8.4 Measurement validity (Anvil mines instantly, so public-chain
> latency is extrapolated, not measured) + §7.3's cents-at-L2-fees framing.

In [ ]:
from a2a_interfaces import fixtures as fx          # the canonical example, one source of truth
from chainmcp.testing import ANVIL_KEYS, anvil_available, artifacts_available, launch_anvil
from chainmcp.client import ChainClient, ChainRevert

CHAIN_OK = anvil_available() and artifacts_available()
SKIP = ("skipped: needs anvil + built contracts — install Foundry "
        "(https://getfoundry.sh), then run:  forge build --root contracts")

anvil = ada = bell = mallory = None
print("anvil on PATH   →", "✓" if anvil_available() else "✗")
print("forge artifacts →", "✓" if artifacts_available() else "✗")
if not CHAIN_OK:
    print(SKIP)

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    STORY_TIME = fx.WINDOW.start - 900        # the chain is born at 13:45, story time
    anvil = launch_anvil(timestamp=STORY_TIME)

    # One client per person. Each holds its OWN key and signs its own transactions —
    # Ada's client cannot spend Bell's money. (These are anvil's public dev keys.)
    ada     = ChainClient(anvil.rpc_url, ANVIL_KEYS["ada"],     deployment=anvil.deployment)
    bell    = ChainClient(anvil.rpc_url, ANVIL_KEYS["bell"],    deployment=anvil.deployment)
    mallory = ChainClient(anvil.rpc_url, ANVIL_KEYS["mallory"], deployment=anvil.deployment)

    print("a fresh private chain at", anvil.rpc_url)
    print("contracts deployed     →", anvil.deployment)
    print("Ada is", ada.address)
    print("Bell is", bell.address)

Those `0x…` strings are **addresses** — account numbers derived from each party's secret
key (chapter 04 shows the derivation). Note they match the canonical fixtures the whole
repo shares: `fx.ADA` *is* Ada's address, everywhere — story, docs, tests, and this cell.

One more piece of table-setting. On-chain TOK is counted in **base units** of
10⁻¹⁸ TOK — chains avoid fractions exactly like your toy did (whole TOK only), they just
pick a much smaller whole unit. So "10 TOK" is the integer 10·10¹⁸. Ada starts broke;
`MockTOK` has a faucet (a "free play-money" tap that only exists on toy tokens):

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    PRICE = int(fx.PRICE_10_TOK)               # "10000000000000000000" = 10 TOK in base units
    ada.faucet(PRICE)                          # Ada taps the play-money faucet

    def tok(units):                            # pretty-printer: base units -> TOK
        return f"{units / 10**18:g} TOK"

    print("price           →", PRICE, "base units =", tok(PRICE))
    print("Ada's balance   →", tok(ada.tok_balance(ada.address)))
    print("Bell's balance  →", tok(bell.tok_balance(bell.address)))

### 4.2 · The canonical deal, for real

`fx.CANONICAL_OFFER` is the very deal you've been simulating: Bell, 50 Mbps
(`capacity_bps=50_000_000` inside `params`), the 14:00–16:00 window, 10 TOK, quote
valid until 14:20, open offer. Bell **signs it** — a real signature this time, made with
Bell's private key inside `chainmcp` — and Ada redeems it at the contract:

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    signed = bell.sign_offer(fx.CANONICAL_OFFER, terms_doc=fx.TERMS_DOC)
    print("Bell's signature:", signed.signature[:24] + "…  (65 bytes — real, not cardboard)")

    tx, ticket_id = ada.approve_and_fulfill(signed)
    # ("approve" is an ERC-20 formality: Ada first grants the machine permission to pull
    #  her TOK — vending-machine etiquette on chains; then fulfill does the atomic swap.)

    print("minted ticket   →", ticket_id)
    print("owner of ticket →", ada.owner_of(ticket_id), " (Ada!)" if ada.owner_of(ticket_id) == fx.ADA else "")
    print("Ada's balance   →", tok(ada.tok_balance(ada.address)))
    print("Bell's balance  →", tok(bell.tok_balance(bell.address)))

Payment moved and the ticket minted — one transaction, atomic, exactly your toy's
`fulfill` with the EVM doing the rollback-safety. (On this *fresh* chain Ada's ticket is
#1; on the story's shared chain, six deals settled first and hers was **#7** — same
machine, busier morning.)

Now the fun part. **Every robbery you performed on the toy, performed on the real
thing** — watch the contract answer with the exact errors you invented:

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    # Robbery 1 — replay: redeem the same signed offer again.
    try:
        ada.approve_and_fulfill(signed)
    except ChainRevert as e:
        print("replay          →", e)

    # Robbery 2 — stale quote: fast-forward the chain past 14:20, try a fresh offer.
    from web3 import Web3
    w3 = Web3(Web3.HTTPProvider(anvil.rpc_url))
    anvil.increase_time(w3, 3 * 3600)                       # jump to ~16:45
    late = fx.CANONICAL_OFFER.model_copy(update={"salt": "0x" + f"{0xB0B:064x}"})
    try:
        ada.approve_and_fulfill(bell.sign_offer(late))
    except ChainRevert as e:
        print("stale quote     →", e)

**✏️ Your turn 5 — tamper with a signed offer**

The signature covers *every field* of the offer. Prove it: take Bell's original signed
offer, lower the price to 9 TOK **after signing** (build a `SignedOffer` whose `offer`
is modified but whose `signature` is Bell's original), and try to redeem it. Predict
the error first. Hints: `fx.CANONICAL_OFFER.model_copy(update={"price": ..., "salt": ...})`
dodges the replay check with a fresh salt; `SignedOffer` is imported for you below.

In [ ]:
from a2a_interfaces.models import SignedOffer

if not CHAIN_OK:
    print(SKIP)
else:
    pass
    # tampered_offer = fx.CANONICAL_OFFER.model_copy(update={...})
    # tampered = SignedOffer(offer=tampered_offer, signature=signed.signature,
    #                        terms_doc=fx.TERMS_DOC)
    # ...try to redeem it as Ada...

<details><summary>✅ Solution 5 — peek only after trying</summary>

```python
tampered_offer = fx.CANONICAL_OFFER.model_copy(
    update={"price": "9000000000000000000", "salt": "0x" + f"{0xBAD:064x}"})
tampered = SignedOffer(offer=tampered_offer, signature=signed.signature,
                       terms_doc=fx.TERMS_DOC)
try:
    ada.approve_and_fulfill(tampered)
except ChainRevert as e:
    print(e)                                   # BadSignature
```

`BadSignature`. Change one bit of any field and Bell's signature no longer matches —
the contract recovers a *stranger's* address from it. This is the property our cardboard
`signed_by` faked, and it's what makes an offer a *promise* rather than a suggestion.
How a signature achieves this is exactly chapter 04.

</details>

### 4.3 · The kill switch, for real

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    # Mallory (not the issuer) reaches for the brake:
    try:
        mallory.revoke(ticket_id)
    except ChainRevert as e:
        print("Mallory revokes →", e)

    # Bell (the issuer) pulls it — twice, because emergencies are messy:
    bell.revoke(ticket_id)
    bell.revoke(ticket_id)
    view = ada.get(ticket_id)
    print("revoked flag    →", view.revoked, "  (and revoking twice was fine)")
    print("ticket still readable — owner:", ada.owner_of(ticket_id))

The ticket survives revocation as evidence, flag flipped, owner unchanged — your §3.6
rules, verbatim. One last inspection: an ERC-721 ticket carries its own "fine print" via
`tokenURI`. This contract renders it **from on-chain storage** on every call, so the
fine print can never 404 and never disagree with the flag:

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    import base64, json
    uri = ada.token_uri(ticket_id)
    fine_print = json.loads(base64.b64decode(uri.split(",", 1)[1]))
    for k, v in fine_print.items():
        print(f"{k:12} {v}")

In [ ]:
# Always clean up your disposable world — never orphan a chain process.
if anvil is not None:
    for client in (ada, bell, mallory):
        client.close()
    anvil.stop()
    print("world ended. (It was disposable — that was the point.)")
else:
    print("nothing to clean up")

## 5 · What you can now say

You built the settlement layer of this project from first principles. In your own words
— and these are sentences you can defend, because you ran the attack that motivates
each one:

- **Why a naive trade fails:** whoever moves first is exposed, and with plain data,
  nothing is even provable afterward.
- **Why the escrow is a program:** a human middleman just relocates the trust problem; a
  smart contract is a vending machine both sides can read and neither can rig.
- **Why each field exists:** `salt` (two same-terms promises must be distinct),
  `valid_until` (quotes go stale; distinct from the service window), `consumer` (quotes
  can be buyer-bound), the signature (offers must be unforgeable promises).
- **Why each check exists:** `OfferAlreadyUsed` (replay), `OfferExpired`,
  `WrongConsumer`, `BadSignature` — you invented all four names before Solidity did.
- **What atomic means:** payment and ticket happen both-or-neither, crash included —
  and the EVM provides that rollback as a platform guarantee, not as contract code.
- **What the entitlement is:** an ERC-721 ticket in a neutral registry — terms on-chain,
  one owner, revocable by flag (issuer-only, idempotent, never deleted: evidence).

**Loose threads, on purpose** — each is a later chapter:

- The signature was cardboard in the toy and magic in the real run. **Chapter 04** opens
  the box: keys, addresses, and why the signing format (EIP-712) is fussier than you'd
  expect.
- A revoked ticket doesn't tear down the *network session* by itself — someone must be
  watching the flag and act. That someone (the controller — never an LLM, always a
  predicate) is **chapter 05**.
- Who turns "ticket says 50 Mbps" into actual router configuration? **Chapter 06**.

## 6 · 📝 For the paper

Where this chapter lands (section numbers follow `main-arxiv.tex`, whose structure the
full report expands): the settlement half of **RQ1**, written up in **§4.3
(entitlement model)** and **§4.4 (smart-contract and NFT lifecycle)**; the contract-layer
rows of the adversarial matrix (**§6.2 → §7.2**); the *when is a token justified*
argument (**§8.3** — the token earns its 268k–447k gas only when the product is a
standing right); and two honest lines in **§8.4 Limitations**. Draft
sentences you can already defend — each paired with the evidence *you ran*:

| you can write… | because you ran… |
|---|---|
| *Payment and entitlement issuance are atomic: both effects occur in a single EVM transaction, and any failed check reverts all of them, so no reachable state exists in which one party holds both the payment and the entitlement.* | §3.5's forced power cut (the toy stranding Ada's money) and §4.2's live `fulfill` |
| *Signed offers are single-use by construction: the contract keys a consumed-ledger by the digest of the entire offer, so redeeming a signed offer twice is rejected regardless of who submits it.* | §3.2's five-tickets-from-one-promise replay, then `OfferAlreadyUsed` from the real contract in §4.2 |
| *The entitlement is an ERC-721 token whose enforceable terms live in contract storage; revocation is an issuer-only, idempotent flag rather than a deletion, so a revoked entitlement remains on-chain evidence of what was promised.* | §3.6's three kill-switch rules, then §4.3, where the revoked ticket stayed owned and readable |
| *The settlement contract deliberately does not police the service window; time-of-use authorization is delegated to a deterministic controller reading chain time.* | §3.4's non-check, buying at 13:45 for a 14:00 window |

**Reviewer objections you can now answer from experience** (not from reading):
*"What if the process crashes mid-swap?"* — the EVM reverts; you watched the version
without that guarantee strand Ada's money. *"What stops replay?"* — the digest ledger;
you ran the replay, twice, on two implementations. *"Can the provider grief the buyer
by revoking instantly?"* — the flag preserves the promise as evidence for a dispute;
compensation is explicitly out of scope.

**Honesty inventory for Limitations** (collected from this chapter's pragmatic 🧭 boxes):
no refund/dispute machinery — revocation is enforcement, not restitution; local devnet
and play-money token — mechanism identical on any EVM chain, but production fee-market
economics are unmeasured here (chapter 09 quantifies what the devnet *can* say).

*Next: [04 — Signatures that contracts believe](04_signatures_that_contracts_believe.ipynb)*